# درس 2: توزیع‌ها: نرمال، لگ‌نرمال و دم پهن

متن کامل درس: [https://quantacademy.ir/courses/quant-math/distributions-fat-tails/](https://quantacademy.ir/courses/quant-math/distributions-fat-tails/)

داده‌های این دوره **شبیه‌سازی‌شده** هستند، نه قیمت واقعی بازار؛ هدف یاد دادن روش است. آموزشی است و توصیه سرمایه‌گذاری نیست.

سلول‌ها را از بالا به پایین اجرا کن (Shift+Enter).

## ۱. توزیع نرمال

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

URL = "https://raw.githubusercontent.com/quantacademyir/quant-math/main/data/btc_daily_5y.csv"
close = pd.read_csv(URL, parse_dates=["date"], index_col="date")["close"]
simple = close.pct_change().dropna()
logret = np.log(close).diff().dropna()
mu, sigma = logret.mean(), logret.std()
print(f"{mu:.5f} {sigma:.4f}")

rng = np.random.default_rng(2)
z = rng.standard_normal(1_000_000)
for k in (1, 2, 3):
    print(k, f"{(np.abs(z) < k).mean():.4f}", f"{stats.norm.cdf(k) - stats.norm.cdf(-k):.4f}")

## ۲. لگ‌نرمال: قیمت

In [ ]:
T = 365
x = rng.normal(mu * T, sigma * np.sqrt(T), size=400_000)
P0 = close.iloc[-1]
P = P0 * np.exp(x)
print(round(P.mean()), round(np.median(P)))
print(round(P0 * np.exp(mu * T + sigma**2 * T / 2)), round(P0 * np.exp(mu * T)))
print(f"{(P > P.mean()).mean():.1%}", f"{stats.skew(P):.2f}", f"{stats.skew(x):.2f}")

## ۳. دم پهن و توزیع t

In [ ]:
print(f"{'df':>4} {'excess_kurtosis':>16} {'P(X<-4sd)':>11}")
print(f"{'norm':>4} {0.0:>16.2f} {stats.norm.cdf(-4):>11.2e}")
for df in (3, 4, 5, 10, 30):
    scale = np.sqrt((df - 2) / df)  # unit variance
    k = 6 / (df - 4) if df > 4 else float("inf")
    p = stats.t.cdf(-4 / scale, df)
    print(f"{df:>4} {k:>16.2f} {p:>11.2e}")

## ۴. داده واقعی

In [ ]:
print(round(stats.kurtosis(logret), 2), round(stats.skew(logret), 2))
df_hat, loc_hat, scale_hat = stats.t.fit(logret)
print(round(df_hat, 2), round(loc_hat, 5), round(scale_hat, 4))

In [ ]:
thr = np.log(0.9)
n = len(simple)
print("<= -10%:", (simple <= -0.10).sum(),
      round(n * stats.norm.cdf(thr, mu, sigma), 2),
      round(n * stats.t.cdf(thr, df_hat, loc_hat, scale_hat), 2))
for k in (3, 4, 5):
    c = (np.abs(logret - mu) > k * sigma).sum()
    expected_normal = len(logret) * 2 * stats.norm.cdf(-k)
    expected_t = len(logret) * 2 * stats.t.cdf(-k * sigma, df_hat, 0, scale_hat)
    print(k, c, round(expected_normal, 2), round(expected_t, 2))

## ۵. QQ به زبان عدد

In [ ]:
zz = (logret - mu) / sigma
qs = [0.001, 0.01, 0.05, 0.95, 0.99, 0.999]
tab = pd.DataFrame({"q": qs, "data": np.quantile(zz, qs), "normal": stats.norm.ppf(qs)})
print(tab.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

## ۶. اثر عملی: ریسک روزانه ۱٪

In [ ]:
var_n = -(mu + sigma * stats.norm.ppf(0.01))
var_h = -np.quantile(logret, 0.01)
var_t = -stats.t.ppf(0.01, df_hat, loc_hat, scale_hat)
print(f"{var_n:.4f} {var_h:.4f} {var_t:.4f}")